**Assets**
- Total assets: book value of all assets
(i.e. intangible and tangible assets, stock, current and non-currents assets)#
- Total liabilities: sum of current liabilities (i.e. loans and short-term debt, creditors and non-current liabilities (i.e. long-term financial liabilities including borrowing from credit institutions and bonds issued).
- Leverage: ratio of total liabilities to total assets.
  
**Income**
- Operating revenue (turnover): sum of net sales, other operating revenues and stock variations.
- Wage bill: renumeration_employees
- Employment: number of employees on the company’s payroll. 
- Negative turnover values. Turnover is defined as the operating revenue in FAME. In a few cases, some companies report negative turnover values. We flag (but keep) those companies reporting negative turnover values.  
   
**Productivity** 
- GVA (Lars): wage bill + EBITDA
- GVA (bottom-up): profit_loss_pretax + interest_paid + depreciation + remuneration_employees
- Productivity: GVA / employees
- Average wage: wage bill / employees
- Use lns

In [ ]:
import ibis
from utils.f_0_dirs import get_data_dirs

old_table_name = "fame_yearly_kp"
new_table_name = "working_yearly"

# Initialize connection
dirs = get_data_dirs()
con = ibis.duckdb.connect(str(dirs.db_path))

# Reference the existing deflated table
working_yearly_kp = con.table(old_table_name)

# Calculate the new metrics using Ibis lazy evaluation
# We use ibis.ifelse to safely handle natural logarithms of negative or zero GVA
working_yearly_expr = working_yearly_kp.mutate(
    gva1 = working_yearly_kp.wages + working_yearly_kp.ebitda,
    gva2 =  working_yearly_kp.profit_loss_pretax +
            working_yearly_kp.interest_paid +
            working_yearly_kp.depreciation +
            working_yearly_kp.remuneration_employees,
    average_wage = working_yearly_kp.wages / working_yearly_kp.employees
).mutate(
    gva1_per_worker = ibis._.gva1 / working_yearly_kp.employees,
    gva2_per_worker = ibis._.gva2 / working_yearly_kp.employees
).select(
    "registered_number", "year",
    "employees", "average_wage", "gva1", "gva2",
    "gva1_per_worker", "gva2_per_worker"
)

print(f"✅ Inserting columns into new '{new_table_name}' table: {working_yearly_expr.columns}")
con.create_table("working_yearly", working_yearly_expr, overwrite=True)

# Verify the final materialized table
final_table = con.table("working_yearly")
row_count = final_table.count().execute()
col_count = len(final_table.columns)

print(f"✅ Materialized 'working_yearly' table.")
print(f"📊 Number of rows: {row_count:,}")
print(f"📊 Number of columns: {col_count}")
print("\nHead of working_yearly:")
display(final_table.sample(200 / row_count).execute())

✅ Inserting columns into new 'working_yearly' table: ('registered_number', 'year', 'gva1', 'gva2', 'gva1_per_worker', 'gva2_per_worker', 'employees', 'average_wage')
✅ Materialized 'working_yearly' table.
📊 Number of rows: 1,128,490
📊 Number of columns: 8

Head of working_yearly:


,registered_number,year,gva1,gva2,gva1_per_worker,gva2_per_worker,employees,average_wage
0,NI012888,2006,2592.197063,NaN,32.402463,NaN,80,27.833366
1,01963821,2006,-12770.365752,NaN,-798.147859,NaN,16,56.461173
2,00621178,2006,4289.095618,NaN,428.909562,NaN,10,111.838452
3,01799580,2006,59393.343031,65804.89517,60.175626,66.671626,987,51.908598
4,02603700,2006,-1959.340542,NaN,-65.311351,NaN,30,51.562642
...,...,...,...,...,...,...,...,...
216,08899140,2024,2815.665000,NaN,24.698816,NaN,114,26.146228
217,08743370,2024,4522.648000,NaN,88.679373,NaN,51,63.417804
218,SC067046,2024,20296.596000,20940.20000,61.691781,63.648024,329,32.756793
219,00688439,2024,10392.000000,NaN,162.375000,NaN,64,30.296875
